In [1]:
import os
import random
import pandas as pd
import shutil
import git
import fnmatch
from tqdm import tqdm

# Change it to your google drive path where this notebook located.
drive_path = '/Users/samyiin/Projects/ZipfLawAnalysis'
os.chdir(drive_path)

In [4]:
df_users = pd.read_csv('Database/TempData/GatherData/Filter_4_SendUserEmail/Results/all_users.csv')

In [5]:
def copy_file_to_dir(original_dir_path, filename, dest_dir_path):
    '''
    By our build, we will assume that filename.py exists in original_dir_path
    and also dest_dir_path exists
    '''
    original_file_path = os.path.join(original_dir_path, filename)
    dest_file_path = os.path.join(dest_dir_path, filename)
    
    # Check for naming conflicts and append (x) if needed
    counter = 1
    while os.path.exists(dest_file_path):
        # split the filename and extension (.py)
        file_name_no_ext, file_ext = os.path.splitext(filename)
        dest_file_path = os.path.join(dest_dir_path, f"{file_name_no_ext}({counter}){file_ext}")
        counter += 1

    # Copy the file to the destination
    try:
        shutil.copy(original_file_path, dest_file_path)    
    except:
        # I met a problem with symbolic link: it only shows when I do ls in terminal, it looks like "cv.py@"
        # we can't do anything about it, it is linking to something else. 
        pass


def save_pyfile_single_project(project_path, dir_save_python_file):
    """
    The following function will walk through a project directory, and find any python files and save them to target directory.

    :param project_path:
    :return:
    """
    list_df_names = []
    for foldername, subfolders, filenames in os.walk(project_path):
        for filename in filenames:
            if fnmatch.fnmatch(filename, '*.py'):
                # save all the python files in this repo and exit. 
                copy_file_to_dir(original_dir_path=foldername, filename=filename, dest_dir_path=dir_save_python_file)
                

def save_pyfile_single_user(df_repos, user_directory_path):
    # if the tempDownload directory exist, then we delete it
    dir_save_python_file = os.path.join(user_directory_path, 'PythonFiles')
    dir_temp_download = os.path.join(user_directory_path, 'TempDownload')
    for directory in [dir_save_python_file, dir_temp_download]:
        if os.path.exists(directory):
            shutil.rmtree(directory)
        os.mkdir(directory)
    list_df_names = []
    # download all the repos for this user
    for i in range(len(df_repos)):
        repo_url = df_repos.iloc[i]['clone_url']
        repo_name = df_repos.iloc[i]['name']
        # create a directory under tempDownload for downloading github repo
        repo_path = os.path.join(dir_temp_download, repo_name)
        if os.path.exists(repo_path):
            shutil.rmtree(repo_path)
        os.mkdir(repo_path)
        # download the project
        try:
            git.Repo.clone_from(repo_url, repo_path)
        except:
            with open('GatherData/Operation_1_DownloadRepos/Temp/missed_projects.txt', "a") as file:
                file.write(repo_url + "\n")
        # project path is always the same because we clone to the same directory
        save_pyfile_single_project(project_path=repo_path, dir_save_python_file=dir_save_python_file)
    
    # delete everything under tempDownload
    shutil.rmtree(dir_temp_download)

def save_pyfile_all_users(df_users):
    for i in tqdm(range(len(df_users))):
        user_login = df_users.iloc[i].to_dict()['login']
        # go to the directory for this user
        user_directory_path = os.path.join('Database/UserData', str(user_login))
        # "Cache" the results: see if this dir is marked as done
        finish_token_fp = os.path.join(user_directory_path, 'finish_OP1')
        if os.path.exists(finish_token_fp):
            # print(f"User {user_login} has already been downloaded.")
            continue
            
        # the df of filtered python repo 
        df_repos = pd.read_csv(os.path.join(os.path.join(user_directory_path, "filtered_python_repos.csv")))
        save_pyfile_single_user(df_repos, user_directory_path)
        
        # finish download files for this user, mark a directory as done: write the finish token
        with open(finish_token_fp, "w") as token_file:
            pass  # Creating an empty file

save_pyfile_all_users(df_users)

100%|███████████████████████████████████████████████████████████████████████████████████████| 576/576 [00:27<00:00, 20.89it/s]


In [ ]:
# should have added project name for more context